In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

In [2]:
import sys
sys.path.append("..")
from src.preprocessing import load_data,build_preprocessor

X,y = load_data()
preprocessor = build_preprocessor(X)

In [3]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline

#RFClassifier model using the preprocessor in pipeline
model_1 = Pipeline(steps=[("preprocessor",preprocessor),("classifier",RandomForestClassifier(random_state=0))])

In [4]:
from sklearn.model_selection import cross_validate
from sklearn.model_selection import StratifiedKFold

#shuffling training data and validation data assigned to each cv
cv_strategy = StratifiedKFold(n_splits=5,shuffle=True,random_state=0)

metrics = ["recall","precision","f1"]


results_model_1 = cross_validate(model_1,X,y,cv=cv_strategy,scoring=metrics,n_jobs=-1,error_score="raise")

for metric in metrics:
    fold_scores = results_model_1[f"test_{metric}"]
    print(f"{metric.capitalize()} per fold:", fold_scores)
    print(f"Mean {metric.capitalize()} per fold:",fold_scores.mean())
    print(f"Standard Deviation of {metric.capitalize()} per fold:",fold_scores.std())


Recall per fold: [0.76797829 0.7642736  0.76574336 0.77026569 0.77037875]
Mean Recall per fold: 0.7677279361965035
Standard Deviation of Recall per fold: 0.002424896974492898
Precision per fold: [0.86965429 0.86789061 0.86922485 0.86338867 0.87013153]
Mean Precision per fold: 0.8680579899032346
Standard Deviation of Precision per fold: 0.0024511158434905884
F1 per fold: [0.8156599  0.81279307 0.81420929 0.81417304 0.81722236]
Mean F1 per fold: 0.8148115328251302
Standard Deviation of F1 per fold: 0.0015083988790110337


In [5]:
from xgboost import XGBClassifier

#Using a new model using xgb
model_2 = Pipeline(steps=[("preprocessor",preprocessor),("classifier",XGBClassifier(random_state=0,eval_metric="logloss"))])

#Model 2 Results
results_model_2 = cross_validate(model_2,X,y,cv=cv_strategy,scoring=metrics,n_jobs=-1,error_score="raise")

for metric in metrics:
    fold_scores = results_model_2[f"test_{metric}"]
    print(f"{metric.capitalize()} per fold:", fold_scores)
    print(f"Mean {metric.capitalize()} per fold:",fold_scores.mean())
    print(f"Standard Deviation of {metric.capitalize()} per fold:",fold_scores.std())

Recall per fold: [0.7230891  0.71192764 0.71758055 0.72402487 0.71701526]
Mean Recall per fold: 0.7187274864691905
Standard Deviation of Recall per fold: 0.00441748826463509
Precision per fold: [0.85380507 0.85025655 0.84446514 0.85159574 0.85173247]
Mean Precision per fold: 0.8503709963400521
Standard Deviation of Precision per fold: 0.003163502490530842
F1 per fold: [0.78302926 0.77496769 0.77586945 0.78264589 0.77858941]
Mean F1 per fold: 0.779020339970718
Standard Deviation of F1 per fold: 0.003339277381639615


In [6]:
# comparison table for both models using two algos
import pandas as pd

comparison = pd.DataFrame({ "Model": ["RandomForest", "XGBoost"], 
                            "Recall": [results_model_1["test_recall"].mean(), results_model_2["test_recall"].mean()], 
                            "Precision": [results_model_1["test_precision"].mean(), results_model_2["test_precision"].mean()], 
                            "F1": [results_model_1["test_f1"].mean(), results_model_2["test_f1"].mean()] })

print(comparison)

          Model    Recall  Precision        F1
0  RandomForest  0.767728   0.868058  0.814812
1       XGBoost  0.718727   0.850371  0.779020


# Putting 2 feature columns for cancel_ratio ie how many guests cancelled and first_timer for guests booking for the 1st time

In [7]:
import numpy as np

X["cancel_ratio"] = np.where((X["previous_cancellations"]==0) & (X["previous_bookings_not_canceled"]==0), 0,X["previous_cancellations"]/(X["previous_bookings_not_canceled"] + X["previous_cancellations"]))
X["first_timer"] = np.where((X["previous_cancellations"]==0)&(X["previous_bookings_not_canceled"]==0),1,0)

In [8]:
preprocessor = build_preprocessor(X)

In [9]:
#RFClassifier model using the preprocessor in pipeline
model_1 = Pipeline(steps=[("preprocessor",preprocessor),("classifier",RandomForestClassifier(random_state=0))])

In [10]:
#shuffling training data and validation data assigned to each cv
cv_strategy = StratifiedKFold(n_splits=5,shuffle=True,random_state=0)

metrics = ["recall","precision","f1"]


results_model_1 = cross_validate(model_1,X,y,cv=cv_strategy,scoring=metrics,n_jobs=-1,error_score="raise")

for metric in metrics:
    fold_scores = results_model_1[f"test_{metric}"]
    print(f"{metric.capitalize()} per fold:", fold_scores)
    print(f"Mean {metric.capitalize()} per fold:",fold_scores.mean())
    print(f"Standard Deviation of {metric.capitalize()} per fold:",fold_scores.std())

Recall per fold: [0.76718679 0.76585642 0.76653477 0.76710006 0.77117015]
Mean Recall per fold: 0.7675696367844728
Standard Deviation of Recall per fold: 0.0018620887805401532
Precision per fold: [0.87087665 0.86924163 0.86978833 0.86820218 0.8701365 ]
Mean Precision per fold: 0.8696490555610061
Standard Deviation of Precision per fold: 0.0008970451674430916
F1 per fold: [0.81574992 0.81428056 0.81490385 0.81452581 0.81766962]
Mean F1 per fold: 0.8154259534988573
Standard Deviation of F1 per fold: 0.0012275004786638077


In [11]:
#Using a new model using xgb
model_2 = Pipeline(steps=[("preprocessor",preprocessor),("classifier",XGBClassifier(random_state=0,eval_metric="logloss"))])

#Model 2 Results
results_model_2 = cross_validate(model_2,X,y,cv=cv_strategy,scoring=metrics,n_jobs=-1,error_score="raise")

for metric in metrics:
    fold_scores = results_model_2[f"test_{metric}"]
    print(f"{metric.capitalize()} per fold:", fold_scores)
    print(f"Mean {metric.capitalize()} per fold:",fold_scores.mean())
    print(f"Standard Deviation of {metric.capitalize()} per fold:",fold_scores.std())

Recall per fold: [0.71811398 0.71271905 0.71859808 0.72470322 0.7209723 ]
Mean Recall per fold: 0.7190213253584077
Standard Deviation of Recall per fold: 0.003919473150702517
Precision per fold: [0.8546629  0.85477966 0.85018727 0.85069675 0.85528433]
Mean Precision per fold: 0.8531221817746424
Standard Deviation of Precision per fold: 0.0022041964139511198
F1 per fold: [0.78046083 0.77731196 0.77887384 0.78266178 0.78240599]
Mean F1 per fold: 0.7803428807149279
Standard Deviation of F1 per fold: 0.0020490082762786937


In [12]:
# comparison table for both models using two algos
import pandas as pd

comparison = pd.DataFrame({ "Model": ["RandomForest", "XGBoost"], 
                            "Recall": [results_model_1["test_recall"].mean(), results_model_2["test_recall"].mean()], 
                            "Precision": [results_model_1["test_precision"].mean(), results_model_2["test_precision"].mean()], 
                            "F1": [results_model_1["test_f1"].mean(), results_model_2["test_f1"].mean()] })

print(comparison)

          Model    Recall  Precision        F1
0  RandomForest  0.767570   0.869649  0.815426
1       XGBoost  0.719021   0.853122  0.780343


# Putting in 2 feature engineering columns moved the results by about 0.01 for both models - not a significant change